## Atelier Préparation des Données

#### Contexte 
Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT.
Chaque capteur 
collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la 
consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de 
fonctionnement et l'état du système de climatisation. 
Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning 
capable de prédire la consommation énergétique ou de détecter les situations anormales. 

Cependant, les données brutes présentent volontairement différents problèmes : valeurs 
manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables 
catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables. 
L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt 
pour le Machine Learning. 

In [61]:
# Importation de librairie
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### <span style="color: #2982fe">Partie 1 – Exploration des données </span>

##### <span style="color: #8eb9f6">1) Chargement des données CSV </span>

In [62]:
df = pd.read_csv("../data/smart_building_raw.csv")

##### <span style="color: #8eb9f6">2) Affichage des premières lignes du dataset</span>

In [63]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


In [64]:
df.shape

(507, 14)

##### <span style="color: #8eb9f6">3) Affichage des dernières lignes du dataset </span>

In [65]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


##### <span style="color: #8eb9f6">4) Nombre d'observations du dataset</span>

In [66]:
print("Nombre de données d'observation : ", df.shape[0])

Nombre de données d'observation :  507


##### <span style="color: #8eb9f6">5) Nombres de variables du dataset</span>

In [67]:
print("Nombre de variables : ", df.shape[1])
print("Nom des variables : ", df.columns.tolist())

Nombre de variables :  14
Nom des variables :  ['id_mesure', 'date', 'batiment', 'type_batiment', 'zone', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


##### <span style="color: #8eb9f6">6) Identifification des variables numériques</span> 

In [68]:
print("Variables de type numérique : ", df.select_dtypes(include=np.number).columns.tolist())

Variables de type numérique :  ['id_mesure', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


##### <span style="color: #8eb9f6">7) Identifification des variables catégorielles</span> 

In [69]:
print('Variables de type catégoriel : ', df.select_dtypes(include='object').columns.tolist())

Variables de type catégoriel :  ['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


##### <span style="color: #8eb9f6">8) Identification des dates</span>

In [70]:
print("Variables date \n: ", df.date)
print("type de la variable date : ", type(df.date[0]))

Variables date 
:  0      2025-02-13 06:00:00
1      2025-03-10 12:00:00
2      2025-05-04 00:00:00
3      2025-01-19 00:00:00
4      2025-04-24 06:00:00
              ...         
502    2025-01-27 12:00:00
503    2025-03-09 12:00:00
504    2025-03-29 00:00:00
505    2025-04-19 18:00:00
506    2025-01-26 12:00:00
Name: date, Length: 507, dtype: object
type de la variable date :  <class 'str'>


##### <span style="color: #8eb9f6">9) Identification des identifiants</span>

In [71]:
print("Les identifiants \n", df.id_mesure)
print("type de la variable id_mesure : ", type(df.id_mesure[0]))
print("Le nombre de identifiants uniques : ", df.id_mesure.nunique())

Les identifiants 
 0      1174
1      1275
2      1493
3      1073
4      1454
       ... 
502    1107
503    1271
504    1349
505    1436
506    1103
Name: id_mesure, Length: 507, dtype: int64
type de la variable id_mesure :  <class 'numpy.int64'>
Le nombre de identifiants uniques :  500


##### <span style="color: #8eb9f6">Informations globales du jeu de données</span>

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_mesure           507 non-null    int64  
 1   date                507 non-null    object 
 2   batiment            507 non-null    object 
 3   type_batiment       503 non-null    object 
 4   zone                507 non-null    object 
 5   temperature         495 non-null    float64
 6   humidite            496 non-null    float64
 7   co2                 500 non-null    float64
 8   occupation          501 non-null    float64
 9   consommation_kwh    502 non-null    float64
 10  mode_climatisation  502 non-null    object 
 11  etat_systeme        507 non-null    object 
 12  jour_semaine        502 non-null    object 
 13  alerte              507 non-null    object 
dtypes: float64(5), int64(1), object(8)
memory usage: 55.6+ KB


##### <span style="color: #8eb9f6">10) Déterminons les statistiques descriptives</span> 

In [73]:
# Statistiques descriptives des variables numériques
df.describe()

,id_mesure,temperature,humidite,co2,occupation,consommation_kwh
count,507.000000,495.000000,496.000000,500.000000,501.000000,502.000000
mean,1251.114398,24.154141,57.864113,844.150000,44.850299,169.069323
std,144.782769,7.418465,16.026336,582.181386,24.949139,53.164294
min,1001.000000,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,1125.500000,21.600000,49.275000,623.750000,27.000000,136.875000
50%,1252.000000,24.000000,57.550000,787.500000,46.000000,169.800000
75%,1376.500000,26.000000,65.750000,952.000000,61.000000,202.975000
max,1500.000000,96.000000,160.000000,6000.000000,116.000000,336.200000


In [74]:
# Statistiques descriptives des variables catégorielles
df.describe(include='object')

,date,batiment,type_batiment,zone,mode_climatisation,etat_systeme,jour_semaine,alerte
count,507,507,503,507,502,507,502,507
unique,500,8,16,4,7,3,7,2
top,2025-04-11 06:00:00,B1,Bureau,B,Normal,Normal,Vendredi,Non
freq,2,94,210,143,271,442,75,361


##### <span style="color: #8eb9f6">11)Variables potentiellement problématiques </span> 

In [75]:
print("type de la variable date : ", type(df.date[0]))

type de la variable date :  <class 'str'>


__Note__ : 
On constate que la variable 'date' est de type 'object' : catégorielle c'est problématique

##### <span style="color: #8eb9f6">12) Pour les données incohérentes</span>
a) recherche des valeurs telles que humidité < 0 

In [76]:
# recherche des valeurs telles que humidité < 0
df[df.humidite < 0]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
281,1126,2025-02-01 06:00:00,B2,École,D,24.8,-5.0,759.0,57.0,225.2,Boost,Normal,Samedi,Non
335,1036,2025-01-09 18:00:00,B4,Bureau,D,21.2,-8.0,160.0,25.0,120.2,Normal,Alerte,Jeudi,Non
366,1216,2025-02-23 18:00:00,B3,Hôpital,B,23.2,-12.0,702.0,41.0,119.8,Normal,Normal,Dimanche,Non


__Note__ : Il y'a trois dates dont la valeur de l'humidité est inférieur à 0

b) recherche des valeurs telles que humidité > 100

In [77]:
# valeurs telles que humidité > 100
df[df.humidite > 100]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
59,1246,2025-03-03 06:00:00,B5,Centre commercial,A,24.2,108.0,813.0,18.0,-15.0,Normal,Normal,Lundi,Non
103,1016,2025-01-04 18:00:00,B6,Université,B,26.9,145.0,670.0,28.0,175.9,Normal,Normal,Samedi,Non
128,1156,2025-02-08 18:00:00,B1,Bureau,A,NaN,160.0,941.0,37.0,149.8,Eco,Normal,Samedi,Oui
160,1186,2025-02-16 06:00:00,B8,Entrepôt,B,24.9,125.0,NaN,18.0,123.5,Normal,Normal,Dimanche,Non
268,1276,2025-03-10 18:00:00,B1,Bureau,D,26.0,140.0,633.0,25.0,148.3,Normal,Normal,Lundi,Non
327,1066,2025-01-17 06:00:00,B2,École,C,25.8,132.0,646.0,14.0,93.6,Eco,Normal,Vendredi,Non
342,1096,2025-01-24 18:00:00,B8,Entrepôt,A,20.3,110.0,783.0,55.0,121.0,Normal,Normal,Vendredi,Non


__Note__ : Il y'a sept dates dont la valeur de l'humidité est supérieur à 100

c) recherche des valeurs telles que température extrêmement élevée 

In [78]:
# valeurs telles que température extrêmement élevée
df[df.temperature > 60]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
7,1141,2025-02-05 00:00:00,B2,École,D,72.5,78.3,257.0,100.0,239.7,Boost,Alerte,Mercredi,Non
116,1181,2025-02-15 00:00:00,B6,Université,A,96.0,86.5,874.0,115.0,241.5,Normal,Normal,Samedi,Non
161,1061,2025-01-16 00:00:00,B5,Centre commercial,D,88.0,50.5,393.0,62.0,196.6,Normal,Alerte,Jeudi,Non
499,1021,2025-01-06 00:00:00,B2,École,A,95.2,65.0,314.0,43.0,246.3,Normal,Normal,Lundi,Non


__Note__ : Il y'a quatre dates dont la valeur de la température est extrémement élevée

d) recherche des valeurs telles occupation que négative 

In [79]:
# Valeurs telles que occupation < 0
df[df.occupation < 0]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
22,1031,2025-01-08 12:00:00,B2,École,D,26.1,57.6,648.0,-5.0,170.0,Eco,Normal,Mercredi,Non
376,1231,2025-02-27 12:00:00,B7,Bureau,D,19.0,35.3,680.0,-8.0,167.9,Eco,Alerte,Jeudi,Non
430,1081,2025-01-21 00:00:00,B1,Bureau,B,23.5,44.1,541.0,-12.0,248.7,Boost,Normal,Mardi,Non
487,1131,2025-02-02 12:00:00,B1,entrepot,A,26.3,70.6,1519.0,-2.0,173.4,Normal,Panne,Dimanche,Oui
494,1331,2025-03-24 12:00:00,B5,Centre commercial,B,22.7,54.6,979.0,-20.0,178.6,Normal,Alerte,Lundi,Oui


__Note__ : Il y'a quatre dates dont la valeur de la variable _occupation_ est négative

e) recherche des valeurs telles que consommation négative

In [80]:
# Valeurs telles que consommation < 0
df[df.consommation_kwh < 0]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
59,1246,2025-03-03 06:00:00,B5,Centre commercial,A,24.2,108.0,813.0,18.0,-15.0,Normal,Normal,Lundi,Non
154,1046,2025-01-12 06:00:00,B2,École,B,21.0,59.4,894.0,33.0,-50.0,Normal,Normal,Dimanche,Non
199,1146,2025-02-06 06:00:00,B7,Bureu,C,32.2,53.8,621.0,103.0,-20.0,Normal,Normal,Jeudi,Non
441,1346,2025-03-28 06:00:00,B5,Centre commercial,D,29.0,44.1,834.0,6.0,-100.0,Eco,Normal,Vendredi,Non


__Note__ : Il y'a quatre dates dont la valeur de la variable _consommation_ est négative

f) Si une valeur est manifestement erronée et qu’on ne peut pas retrouver sa vraie valeur, la 
transformer en valeur manquante 

In [81]:
# Remplacement des valeurs érronnées par des NaN

df["humidite"] = df["humidite"].mask(df["humidite"] < 0)
df["humidite"] = df["humidite"].mask(df["humidite"] > 100)

df["consommation_kwh"] = df["consommation_kwh"].mask(df["consommation_kwh"] < 0)
df["occupation"] = df["occupation"].mask(df["occupation"] < 0)
df["temperature"] = df["temperature"].mask(df["temperature"] > 60)


g) rechercher des valeurs telles que catégories mal orthographiées.

In [82]:
# Les variables catégorielles
var_cats = df.select_dtypes(include='object').columns.tolist()
print("Variables catégorielles : ", var_cats)

Variables catégorielles :  ['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


In [83]:
var_cats = variables_catego.drop(columns=['date']).select_dtypes(include='object').columns.tolist() 

NameError: name 'variables_catego' is not defined

In [ ]:
for var in var_cats:
    print(f"Variable : {var}")
    print(df[var].unique())
    print("\n")

Variable : batiment
['B8' 'B7' 'B5' 'B6' 'B3' 'B4' 'B2' 'B1']


Variable : type_batiment
['Entrepôt' 'Bureau' 'Centre commercial' 'Université' 'Hôpital' 'École'
 'ÉCOLE' 'ecole' 'BUREAU' nan 'Bureu' 'bureau' ' UNIVERSITÉ' 'hôpital '
 'centre commercial' ' Bureau ' 'entrepot']


Variable : zone
['A' 'D' 'B' 'C']


Variable : mode_climatisation
['Eco' 'Normal' 'Boost' nan 'normal' 'BOOST' 'normale' 'Normal ']


Variable : etat_systeme
['Normal' 'Alerte' 'Panne']


Variable : jour_semaine
['Jeudi' 'Lundi' 'Dimanche' 'Vendredi' 'Mercredi' 'Mardi' 'Samedi' nan]


Variable : alerte
['Non' 'Oui']




In [ ]:
# Supprime les espaces au début/fin et passe tout en minuscules
df['etat_systeme'] = df['etat_systeme'].str.strip().str.lower()
df['etat_systeme'].unique()

array(['normal', 'alerte', 'panne'], dtype=object)

In [ ]:
cols_cib = df[['mode_climatisation', 'etat_systeme', 'jour_semaine', 'type_batiment', 'alerte']]

In [ ]:
for col in cols_cib:
    cols_cib[col] = cols_cib[col].str.strip().str.lower()

C:\Users\cissc\AppData\Local\Temp\ipykernel_18776\1765834368.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cols_cib[col] = cols_cib[col].str.strip().str.lower()


In [ ]:
cols_cib

,mode_climatisation,etat_systeme,jour_semaine,type_batiment,alerte
0,eco,normal,jeudi,entrepôt,non
1,eco,normal,lundi,bureau,non
2,normal,normal,dimanche,centre commercial,oui
3,normal,normal,dimanche,université,non
4,normal,normal,jeudi,bureau,non
...,...,...,...,...,...
502,normal,normal,lundi,école,non
503,eco,alerte,dimanche,hôpital,oui
504,eco,normal,samedi,centre commercial,non
505,boost,normal,samedi,bureau,oui


In [ ]:
df[cols_cib.columns] = cols_cib
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,entrepôt,A,23.0,40.0,851.0,26.0,138.4,eco,normal,jeudi,non
1,1275,2025-03-10 12:00:00,B7,bureau,D,28.0,69.0,538.0,0.0,170.2,eco,normal,lundi,non
2,1493,2025-05-04 00:00:00,B5,centre commercial,B,19.5,52.3,1198.0,64.0,149.8,normal,normal,dimanche,oui
3,1073,2025-01-19 00:00:00,B6,université,D,20.0,58.4,1014.0,30.0,141.6,normal,normal,dimanche,non
4,1454,2025-04-24 06:00:00,B7,bureau,D,26.8,70.5,628.0,42.0,228.3,normal,normal,jeudi,non


13) Traitement des valeurs manquantes : 

a) Calcule du nombre et le pourcentage de valeurs manquantes par colonne

In [ ]:
# Nombre de valeurs manquantes par variable
df.isnull().sum()

id_mesure              0
date                   0
batiment               0
type_batiment          4
zone                   0
temperature           16
humidite              21
co2                    7
occupation            11
consommation_kwh       9
mode_climatisation     5
etat_systeme           0
jour_semaine           5
alerte                 0
dtype: int64

In [ ]:
# Nombre total de valeurs manquantes dans le DataFrame
print("Nombre total de valeurs manquantes :", df.isnull().sum().sum())

Nombre total de valeurs manquantes : 0


In [ ]:
# Pourcentage de valeurs manquantes par variable avec 2 chiffres après la virgule
(df.isnull().mean()*100).round(2)

id_mesure             0.00
date                  0.00
batiment              0.00
type_batiment         0.79
zone                  0.00
temperature           3.16
humidite              4.14
co2                   1.38
occupation            2.17
consommation_kwh      1.78
mode_climatisation    0.99
etat_systeme          0.00
jour_semaine          0.99
alerte                0.00
dtype: float64

b) Quelle variable possède le plus de valeurs manquantes ? 

__Note__ : On voit bien que la colonne humidite posséde plus de valeures manquantes avec 4.14 %

In [ ]:
# Pourcentage de valeurs manquantes par variable avec 2 chiffres après la virgule
print("Pourcentage total de valeurs manquantes :", (df.isnull().mean()*100).sum().round(2))

Pourcentage total de valeurs manquantes : 15.38


c) Quelle stratégie utiliser pour les valeurs manquantes ?

__Note__ : Comme on posséde un jeux de données qui ne contient que 500 lignes, supprimer 15% les valeurs manquantes serait un perte conséquente d'information. Dans ce cas on choisit de les remplacer par la moyenne (médiane) pour les variables numériques et la mode pour les variables catégorielles


d) Peut-on supprimer toutes les lignes contenant des valeurs manquantes ? 

Non.

Comme on posséde un jeux de données qui ne contient que 500 lignes, supprimer 15% les valeurs manquantes serait un perte conséquente d'information.

e) Dans quels cas utiliser la moyenne ? 

On utilise souvent la moyenne si on n'a pas des valeurs extremes

f) Quand préférer la médiane ? 

On utilise souvent la médiane quand on a des valeures extrémes qui peut fausser la moyenne

In [ ]:
# Nombre total de valeurs manquantes dans le DataFrame aprés le remplacement
print("Nombre total de valeurs manquantes :", df.isnull().sum().sum())

Nombre total de valeurs manquantes : 0


g) traitement une variable catégorielle ? 

Pour les variables catégorielles on choisit plus souvent de remplacer les valeurs manquantes par la mode

In [ ]:
# Remplacement des valeurs manquantes par la moyenne pour les variables numériques et par le mode pour les variables catégorielles
for column in df.columns:
    if df[column].dtype == 'object':
        df[column].fillna(df[column].mode()[0], inplace=True)
    else:
        df[column].fillna(df[column].mean(), inplace=True)

C:\Users\cissc\AppData\Local\Temp\ipykernel_18776\2751034278.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column].fillna(df[column].mean(), inplace=True)
C:\Users\cissc\AppData\Local\Temp\ipykernel_18776\2751034278.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

##### <span style="color: #8eb9f6">14) Pour les doublons</span>

a) Identification

In [ ]:
df.duplicated().sum()

np.int64(7)

In [ ]:
# Les dupliqués
duplicates = df[df.duplicated(keep=False)]
duplicates.shape

(14, 14)

b) Affichage

In [ ]:
duplicates

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
9,1118,2025-01-30 06:00:00,B1,Bureau,A,22.0,NaN,479.0,64.0,145.0,Normal,Normal,Jeudi,Non
23,1264,2025-03-07 18:00:00,B1,Bureau,B,21.1,38.3,1034.0,74.0,199.2,Normal,Panne,Vendredi,Oui
27,1456,2025-04-24 18:00:00,B5,Centre commercial,D,26.8,61.3,843.0,59.0,186.9,Normal,Normal,Jeudi,Non
56,1479,2025-04-30 12:00:00,B3,Hôpital,B,23.9,68.9,644.0,27.0,162.5,Normal,Normal,Mercredi,Non
117,1026,2025-01-07 06:00:00,B6,BUREAU,A,23.0,74.3,952.0,89.0,247.6,Normal,Normal,Mardi,Oui
123,1118,2025-01-30 06:00:00,B1,Bureau,A,22.0,NaN,479.0,64.0,145.0,Normal,Normal,Jeudi,Non
192,1026,2025-01-07 06:00:00,B6,BUREAU,A,23.0,74.3,952.0,89.0,247.6,Normal,Normal,Mardi,Oui
238,1402,2025-04-11 06:00:00,B2,École,A,22.6,44.0,868.0,62.0,172.5,Eco,Normal,Vendredi,Non
362,1456,2025-04-24 18:00:00,B5,Centre commercial,D,26.8,61.3,843.0,59.0,186.9,Normal,Normal,Jeudi,Non
418,1402,2025-04-11 06:00:00,B2,École,A,22.6,44.0,868.0,62.0,172.5,Eco,Normal,Vendredi,Non


c) sont-ils réellement identiques ?

In [ ]:
doublons_tries = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
doublons_tries

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
117,1026,2025-01-07 06:00:00,B6,BUREAU,A,23.0,74.3,952.0,89.0,247.6,Normal,Normal,Mardi,Oui
192,1026,2025-01-07 06:00:00,B6,BUREAU,A,23.0,74.3,952.0,89.0,247.6,Normal,Normal,Mardi,Oui
9,1118,2025-01-30 06:00:00,B1,Bureau,A,22.0,NaN,479.0,64.0,145.0,Normal,Normal,Jeudi,Non
123,1118,2025-01-30 06:00:00,B1,Bureau,A,22.0,NaN,479.0,64.0,145.0,Normal,Normal,Jeudi,Non
23,1264,2025-03-07 18:00:00,B1,Bureau,B,21.1,38.3,1034.0,74.0,199.2,Normal,Panne,Vendredi,Oui
434,1264,2025-03-07 18:00:00,B1,Bureau,B,21.1,38.3,1034.0,74.0,199.2,Normal,Panne,Vendredi,Oui
455,1320,2025-03-21 18:00:00,B7,Bureau,B,27.5,27.2,359.0,33.0,185.8,Normal,Normal,Vendredi,Non
464,1320,2025-03-21 18:00:00,B7,Bureau,B,27.5,27.2,359.0,33.0,185.8,Normal,Normal,Vendredi,Non
238,1402,2025-04-11 06:00:00,B2,École,A,22.6,44.0,868.0,62.0,172.5,Eco,Normal,Vendredi,Non
418,1402,2025-04-11 06:00:00,B2,École,A,22.6,44.0,868.0,62.0,172.5,Eco,Normal,Vendredi,Non


In [ ]:
doublons_tries.shape

(14, 14)

__Note__ : Oui Les doublons sont réellement tous identiques 2 à 2

d) suppression des doublons réellement identifiés. 

In [85]:
df = df.drop_duplicates(keep='first')

e) Vérification de la suppression 

In [86]:
# verification de la suppression des doublons
df.duplicated().sum()

np.int64(0)